# Medical Transcription & SOAP Note Generation

End-to-end pipeline for the technical assessment:

1. **Part A - Speech-to-Text:** load `audio/sample_dictation.mp3` and produce a verbatim transcript.
2. **Part B - Text-to-SOAP:** structure the transcript into Subjective, Objective, Assessment, and Plan.
3. **Bonus:** run a cross-contamination check that flags patient-reported content in Objective (and clinician-observed content in Subjective).

### Tool selection

| Component | Tool | Rationale |
|---|---|---|
| Speech-to-text (preferred) | **faster-whisper** (CTranslate2) | Runs locally so PHI never leaves the machine. INT8 CPU inference is ~4x faster than openai-whisper with the same weights. |
| Speech-to-text (fallback) | **Gemini multimodal API** | Works on CPU-only machines with no model download; handles 13+ minute audio. |
| SOAP structuring | **Gemini** (`gemini-3.5-flash-lite`) | Strong clinical entity recognition, native JSON response mode, low temperature for reproducible section assignment. |

> **Privacy note:** the code prefers local Whisper. The committed transcript was generated with the Gemini fallback because faster-whisper was not installed in the build environment; set `backend="local"` after installing `faster-whisper` to reproduce fully offline. See the README for details.


## 0. Setup

Dependencies: `pip install -r requirements.txt` (or `pip install google-genai faster-whisper`).
Set `GEMINI_API_KEY` before running the SOAP step.


In [1]:
import json
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

# Make the project modules importable when the notebook is launched from the repo root.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s")
logger = logging.getLogger("notebook")

from config import (
    API_MAX_RETRIES,
    DEFAULT_AUDIO_FILE,
    GEMINI_API_KEY,
    GEMINI_MODEL,
    GEMINI_TEMPERATURE,
    GEMINI_TRANSCRIPTION_MODEL,
    OUTPUT_DIR,
    WHISPER_BEAM_SIZE,
    WHISPER_DEVICE,
    WHISPER_LANGUAGE,
    WHISPER_MODEL_SIZE,
)
from medical_lexicon import normalize_medical_terms
from soap_generator import SOAPNoteGenerator
from transcriber import get_transcriber

print(f"Project root : {PROJECT_ROOT}")
print(f"Audio file   : {DEFAULT_AUDIO_FILE}")
print(f"Audio exists : {DEFAULT_AUDIO_FILE.exists()}")
print(f"Audio size   : {DEFAULT_AUDIO_FILE.stat().st_size / 1024 / 1024:.2f} MB")
print(f"API key set  : {bool(GEMINI_API_KEY)}")


Project root : D:\Technical Assignment\medical-transcription-soap
Audio file   : D:\Technical Assignment\medical-transcription-soap\audio\sample_dictation.mp3
Audio exists : True
Audio size   : 4.23 MB
API key set  : True


In [2]:
# Audio duration is useful context for reading the transcript.
if shutil.which("ffprobe"):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(DEFAULT_AUDIO_FILE)],
        capture_output=True, text=True, check=True,
    )
    seconds = float(result.stdout.strip())
    print(f"Duration: {int(seconds // 60)} min {seconds % 60:04.1f} s")
else:
    print("ffprobe not available; skipping duration check.")


Duration: 13 min 45.0 s


## 1. Part A - Audio transcription

`get_transcriber(backend="auto")` attempts local faster-whisper first and falls back to Gemini when the package or model is unavailable. Both backends are asked for a verbatim transcript, including disfluencies, so the output can be compared with a human ground truth.

After speech-to-text, a deliberately small medical lexicon fixes unambiguous ASR errors (for example `supernate -> supinate`, `scolidal -> skeletal`). The uncorrected ASR text is preserved separately for audit.

> **Reproducibility toggle:** a live cloud ASR call is non-deterministic, so the committed notebook was executed with `USE_SAVED_TRANSCRIPT=1`. That reuses `output/transcript_raw.txt` (the verbatim ASR output shipped with the repo) and still runs the full normalisation + SOAP steps. Clear the variable and rerun to transcribe `audio/sample_dictation.mp3` from scratch.


In [3]:
# Set USE_SAVED_TRANSCRIPT=1 to reuse the committed raw ASR transcript.
# Leave it unset to run speech-to-text live against audio/sample_dictation.mp3.
use_saved = os.environ.get("USE_SAVED_TRANSCRIPT") == "1"
raw_path = OUTPUT_DIR / "transcript_raw.txt"

if use_saved and raw_path.exists():
    raw_transcript = raw_path.read_text(encoding="utf-8")
    print("Loaded committed raw transcript (USE_SAVED_TRANSCRIPT=1).")
    print("Unset the variable and rerun this cell to transcribe the audio again.")
else:
    transcriber = get_transcriber(
        backend="auto",                  # local first, Gemini fallback
        whisper_model=WHISPER_MODEL_SIZE,
        gemini_api_key=GEMINI_API_KEY,
        gemini_model=GEMINI_TRANSCRIPTION_MODEL,
        whisper_device=WHISPER_DEVICE,
        whisper_beam_size=WHISPER_BEAM_SIZE,
        whisper_language=WHISPER_LANGUAGE,
        max_retries=API_MAX_RETRIES,
    )
    print(f"Backend selected: {transcriber.name}")
    raw_transcript = transcriber.transcribe(DEFAULT_AUDIO_FILE)

transcript, changes = normalize_medical_terms(raw_transcript)

print(f"Raw transcript : {len(raw_transcript):,} characters")
print(f"Terminology fixes: {changes or 'none'}")
print("\n--- Transcript preview (first 1,200 characters) ---\n")
print(transcript[:1200] + (" ..." if len(transcript) > 1200 else ""))


Loaded committed raw transcript (USE_SAVED_TRANSCRIPT=1).
Unset the variable and rerun this cell to transcribe the audio again.
Raw transcript : 8,310 characters
Terminology fixes: [('supernate', 'supinate'), ('adducing', 'adducting'), ('scolidal', 'skeletal'), ('throbby', 'throbbing')]

--- Transcript preview (first 1,200 characters) ---

Hi. I've I've had this pain on the outside of my uh right elbow now. It's it's I first started know noticing it several months ago, but recently it's just been more painful. Okay. So you said several months ago. Um did anything happen several months ago? Was there any sort of trigger, trauma, anything like that to that area? No, there wasn't any any trauma or any uh triggers that that I noticed. I was just um feeling it uh a bit more at the end of of work um uh yeah, I was just getting having it a feeling the pain a bit more at the end of work. Okay. Does uh anything make it better or worse, the pain? Um yeah, if I really if I'm just resting the the 

In [4]:
# Persist both the auditable raw ASR output and the normalised transcript.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
raw_path = OUTPUT_DIR / "transcript_raw.txt"
transcript_path = OUTPUT_DIR / "transcript.txt"

raw_path.write_text(raw_transcript, encoding="utf-8")
transcript_path.write_text(transcript, encoding="utf-8")

print(f"Saved raw transcript      -> {raw_path}")
print(f"Saved final transcript    -> {transcript_path}")


Saved raw transcript      -> D:\Technical Assignment\medical-transcription-soap\output\transcript_raw.txt
Saved final transcript    -> D:\Technical Assignment\medical-transcription-soap\output\transcript.txt


## 2. Part B - SOAP note generation

The system prompt is the core of the structuring step. It:

* Declares the transcript the **only source of truth** (no invented vitals, labs, medicines, or follow-up intervals).
* Defines strict boundaries for S (patient-reported) versus O (clinician-observed/tested).
* Treats guided self-palpation during a telehealth visit as an examination finding.
* Requires pertinent negatives and exact laterality/numbers.
* Requests JSON via `response_mime_type="application/json"` with temperature `0.2`.


In [5]:
generator = SOAPNoteGenerator(
    api_key=GEMINI_API_KEY,
    model=GEMINI_MODEL,
    temperature=GEMINI_TEMPERATURE,
    max_retries=API_MAX_RETRIES,
)
soap_note = generator.generate(transcript)
print(soap_note.to_json())


INFO    | Generating SOAP note with Gemini (gemini-3.5-flash-lite)...


INFO    | AFC is enabled with max remote calls: 10.


INFO    | HTTP Request: POST https://generativelanguage.googleapis.com/v1/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"


INFO    | SOAP validation passed: no cross-contamination detected.


INFO    | SOAP note generation complete.


{
  "subjective": "- Chief complaint: Pain on the outside of the right elbow, starting several months ago and recently more painful, getting worse.\n- Onset: First noticed several months ago, no trauma or specific triggers reported.\n- Aggravating factors: End of work, bending wrist down, picking things up, holding heavy objects at work (does repetitive line work), right-handed.\n- Relieving factors: Resting the elbow, ibuprofen, drugstore topical ointment.\n- Pain quality: Achy, and sometimes sharp depending on movement.\n- Severity: Rated as 4 out of 10.\n- Pertinent negatives: No previous experience with this type of pain, no aches or pains in other joints or muscle groups, no pain radiation, no numbness or tingling, no arm feeling cold, no other symptoms like weight loss, fever, chills, fatigue, rashes, sensory changes, respiratory symptoms, cardiac symptoms, gastrointestinal symptoms, urinary symptoms, or memory loss/confusion.\n- Past medical history: Otherwise healthy.\n- Past s

### 2.1 Bonus - cross-contamination check

`SOAPNoteGenerator._validate_soap()` scans each section for language belonging to the other. It emits warnings rather than blocking the note, because a phrase such as "full range of motion" can legitimately be patient-reported. The check is a safety net for human review, not a semantic classifier.


In [6]:
issues = generator.last_validation_issues
if issues:
    print("Cross-contamination warnings:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("PASS: no cross-contamination warnings.")


PASS: no cross-contamination warnings.


In [7]:
display(Markdown(soap_note.to_markdown()))


# SOAP Note

## Subjective
- Chief complaint: Pain on the outside of the right elbow, starting several months ago and recently more painful, getting worse.
- Onset: First noticed several months ago, no trauma or specific triggers reported.
- Aggravating factors: End of work, bending wrist down, picking things up, holding heavy objects at work (does repetitive line work), right-handed.
- Relieving factors: Resting the elbow, ibuprofen, drugstore topical ointment.
- Pain quality: Achy, and sometimes sharp depending on movement.
- Severity: Rated as 4 out of 10.
- Pertinent negatives: No previous experience with this type of pain, no aches or pains in other joints or muscle groups, no pain radiation, no numbness or tingling, no arm feeling cold, no other symptoms like weight loss, fever, chills, fatigue, rashes, sensory changes, respiratory symptoms, cardiac symptoms, gastrointestinal symptoms, urinary symptoms, or memory loss/confusion.
- Past medical history: Otherwise healthy.
- Past surgical history: C-section for son about 17 years ago (not an emergency).
- Medications: Takes Tylenol and Advil/ibuprofen for pain.
- Allergies: No allergies.
- Social history: Does not smoke, does not use marijuana, drinks about one glass of wine a week, no sick contacts, no recent travel.
- Family history: No family history of skeletal/muscular issues, neurological issues, or autoimmune conditions.

## Objective
- General appearance: Not documented in the provided dictation.
- Inspection of right elbow: No redness or swelling, appears symmetrical and the same compared to other elbow, no skin changes or deformities, area does not feel hot.
- Palpation: Pain at the lateral elbow (lateral epicondyle), no pain at the medial aspect, forearm, or biceps.
- Range of motion and strength testing (self-examination/guided): Full range of motion of both the elbow and hand. Pain with wrist extension with elbow in full extension. Pain with wrist flexion with elbow in full extension. No pain with elbow flexion. Shoulder extension, adduction, and forearm pronation and supination feel normal and cause no pain.

## Assessment
- Lateral epicondylitis (also known as tennis elbow).

## Plan
- Imaging: Not deemed necessary (no trauma, low concern for fracture).
- Activity modifications: Avoid overly repetitive work activities, especially those using extensor forearm muscles.
- Support: Use a compression sleeve.
- Medications: Continue taking Tylenol or NSAIDs for pain.
- Physical therapy: Recommended to increase mobility and strength in the arm.
- Additional measures: Apply ice for pain.
- Prognosis: Should resolve in about six months to two years at most, normally goes away in a couple of months with physical therapy, compression, and splints.


In [8]:
json_path = OUTPUT_DIR / "soap_note.json"
md_path = OUTPUT_DIR / "soap_note.md"

json_path.write_text(soap_note.to_json(), encoding="utf-8")
md_path.write_text(soap_note.to_markdown(), encoding="utf-8")

print(f"Saved SOAP JSON -> {json_path}")
print(f"Saved SOAP MD   -> {md_path}")
for path in (raw_path, transcript_path, json_path, md_path):
    print(f"  {path.name:20s} {path.stat().st_size:6,d} bytes")


Saved SOAP JSON -> D:\Technical Assignment\medical-transcription-soap\output\soap_note.json
Saved SOAP MD   -> D:\Technical Assignment\medical-transcription-soap\output\soap_note.md
  transcript_raw.txt    8,310 bytes
  transcript.txt        8,312 bytes
  soap_note.json        2,787 bytes
  soap_note.md          2,789 bytes


## 3. Reproducing and scaling

**CLI (same logic, scriptable):**
```bash
python pipeline.py                       # auto backend, committed sample audio
python pipeline.py --backend gemini      # force cloud transcription
python pipeline.py --backend local --whisper-model small
python pipeline.py --output-dir runs/exp-1
```

**Tests:**
```bash
pip install -r requirements-dev.txt
pytest -q
```

### Scaling notes

| Concern | Approach |
|---|---|
| PHI / HIPAA | Run faster-whisper (`large-v3`) on an on-prem GPU node. Swap Gemini for a local instruction-tuned model served by vLLM/Ollama behind the same `SOAPNoteGenerator` interface. |
| Throughput | Decouple upload/transcription/structuring with a queue (Celery/RabbitMQ/Redis) so long dictations do not block HTTP requests. |
| EHR integration | Map the JSON to HL7 FHIR `DocumentReference`/`Composition`; keep the audit trail (raw ASR + prompt version + model version). |
| Multi-party audio | Add `pyannote.audio` diarization before structuring; feed speaker roles into the prompt. |
| Quality monitoring | Track word error rate on a de-identified gold set and manually review all cross-contamination warnings. |
